# ¿Las canciones tristes arrasan más en invierno?
## Análisis de valence, popularidad y estacionalidad en Spotify

**Dilema:** Spotify asigna a cada canción un `valence` (0 = muy triste, 1 = muy alegre).
¿Las canciones con valence bajo son más populares en los meses de invierno?
¿O escuchamos lo mismo independientemente del frío?

**Métodos:**
- EDA de audio features
- Análisis estacional de valence
- Correlación Pearson: valence vs popularidad
- Test Mann-Whitney: invierno vs verano
- Clustering K-Means por emoción (valence × energy)

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Estilo
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'text.color':       '#eee',
    'grid.color':       '#2a2a2a',
    'grid.linewidth':   0.6,
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.labelsize':   10,
})

PINK   = '#f4a7b9'
BLUE   = '#7eb8f7'
PURPLE = '#c4a7e7'
GREEN  = '#9ece6a'

SEED = 42
np.random.seed(SEED)
print('Setup OK')

## 1. Datos

In [ ]:
DATA_DIR = Path('../data')

def load_real_data():
    """Intenta cargar el dataset de Kaggle o el generado por fetch_data.py."""
    for f in DATA_DIR.glob('*.csv'):
        df = pd.read_csv(f)
        required = {'valence', 'energy', 'popularity'}
        if required.issubset(df.columns):
            print(f'Dataset real cargado: {f.name} ({len(df):,} filas)')
            return df
    return None


def generate_synthetic():
    """
    Dataset sintético realista.
    Hipótesis embedded: las canciones con valence bajo son ligeramente
    más populares en invierno (noviembre–febrero).
    """
    n = 8000
    genres = {
        'sad':        {'valence_mu': 0.18, 'valence_sd': 0.08, 'energy_mu': 0.35, 'pop_base': 52},
        'melancholy':  {'valence_mu': 0.25, 'valence_sd': 0.10, 'energy_mu': 0.40, 'pop_base': 48},
        'blues':       {'valence_mu': 0.30, 'valence_sd': 0.10, 'energy_mu': 0.45, 'pop_base': 42},
        'emo':         {'valence_mu': 0.22, 'valence_sd': 0.09, 'energy_mu': 0.65, 'pop_base': 55},
        'happy':       {'valence_mu': 0.78, 'valence_sd': 0.09, 'energy_mu': 0.75, 'pop_base': 60},
        'dance':       {'valence_mu': 0.72, 'valence_sd': 0.10, 'energy_mu': 0.82, 'pop_base': 65},
        'pop':         {'valence_mu': 0.58, 'valence_sd': 0.15, 'energy_mu': 0.68, 'pop_base': 62},
        'latin':       {'valence_mu': 0.65, 'valence_sd': 0.12, 'energy_mu': 0.72, 'pop_base': 63},
        'acoustic':    {'valence_mu': 0.45, 'valence_sd': 0.14, 'energy_mu': 0.38, 'pop_base': 50},
        'folk':        {'valence_mu': 0.40, 'valence_sd': 0.12, 'energy_mu': 0.35, 'pop_base': 44},
        'metal':       {'valence_mu': 0.28, 'valence_sd': 0.10, 'energy_mu': 0.88, 'pop_base': 45},
        'rock':        {'valence_mu': 0.48, 'valence_sd': 0.14, 'energy_mu': 0.78, 'pop_base': 55},
    }

    rows = []
    for genre, p in genres.items():
        ng = n // len(genres)
        months = np.random.choice(range(1, 13), ng)
        
        # Efecto estacional en valence (invierno = canciones más tristes)
        season_bias = np.where(np.isin(months, [11, 12, 1, 2]), -0.06, 0.03)
        
        valence = np.clip(
            np.random.normal(p['valence_mu'], p['valence_sd'], ng) + season_bias, 0, 1
        )
        energy = np.clip(np.random.normal(p['energy_mu'], 0.12, ng), 0, 1)
        
        # Popularidad: base + ruido + leve penalización a valence muy bajo o muy alto
        pop_noise = np.random.normal(0, 12, ng)
        pop_valence_effect = -8 * (valence - 0.5) ** 2  # pico en valence medio
        pop_winter_effect  = np.where(np.isin(months, [11, 12, 1, 2]),
                                      -5 * valence + 4, 0)  # invierno favorece tristeza
        popularity = np.clip(
            p['pop_base'] + pop_noise + pop_valence_effect + pop_winter_effect, 0, 100
        ).astype(int)

        for i in range(ng):
            year = np.random.choice(range(2018, 2025))
            rows.append({
                'track_name':    f'{genre}_track_{i}',
                'artist':        f'Artist_{np.random.randint(1, 500)}',
                'genre':         genre,
                'release_date':  f'{year}-{months[i]:02d}-01',
                'month':         months[i],
                'year':          year,
                'valence':       round(valence[i], 4),
                'energy':        round(energy[i], 4),
                'danceability':  round(np.clip(np.random.normal(0.55, 0.18), 0, 1), 4),
                'acousticness':  round(np.clip(np.random.beta(2, 5), 0, 1), 4),
                'tempo':         round(np.random.normal(120, 28), 1),
                'popularity':    int(popularity[i]),
            })

    df = pd.DataFrame(rows)
    print(f'Dataset sintético generado: {len(df):,} tracks, {len(genres)} géneros')
    return df


df = load_real_data()
if df is None:
    print('Sin datos reales — usando dataset sintético')
    df = generate_synthetic()

# Parsear fecha y extraer mes/año
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
if 'month' not in df.columns:
    df['month'] = df['release_date'].dt.month
if 'year' not in df.columns:
    df['year'] = df['release_date'].dt.year

# Etiqueta de estación
season_map = {12:'Invierno', 1:'Invierno', 2:'Invierno',
              3:'Primavera', 4:'Primavera', 5:'Primavera',
              6:'Verano',   7:'Verano',   8:'Verano',
              9:'Otoño',   10:'Otoño',  11:'Otoño'}
df['season'] = df['month'].map(season_map)

print(df[['genre','month','valence','energy','popularity']].head())
print(f'\nShape: {df.shape}')

## 2. EDA — ¿Cómo suenan los datos?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('EDA — Distribución de audio features', fontsize=14, y=1.01)

features = [('valence', PINK, '← triste / alegre →'),
            ('energy',  BLUE, '← suave / intenso →'),
            ('popularity', GREEN, 'popularidad Spotify (0–100)')]

for ax, (feat, color, xlabel) in zip(axes, features):
    data = df[feat].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.85, edgecolor='none')
    ax.axvline(data.mean(), color='white', lw=1.5, ls='--', label=f'media={data.mean():.2f}')
    ax.set_title(feat)
    ax.set_xlabel(xlabel)
    ax.legend(fontsize=8)
    ax.grid(axis='y')

plt.tight_layout()
plt.savefig('../data/img_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Análisis estacional — valence por mes

In [ ]:
monthly = df.groupby('month').agg(
    valence_mean=('valence', 'mean'),
    valence_se=('valence', lambda x: x.std() / np.sqrt(len(x))),
    popularity_mean=('popularity', 'mean'),
    n=('valence', 'count'),
).reset_index()

month_names = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
monthly['month_name'] = [month_names[m-1] for m in monthly['month']]

# Color por estación
season_colors = {'Invierno': BLUE, 'Primavera': GREEN, 'Verano': PINK, 'Otoño': PURPLE}
bar_colors = [season_colors[season_map[m]] for m in monthly['month']]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('¿Las canciones tristes arrasan más en invierno?', fontsize=14)

# Panel 1: valence por mes
bars = ax1.bar(monthly['month_name'], monthly['valence_mean'],
               color=bar_colors, alpha=0.9, zorder=2)
ax1.errorbar(monthly['month_name'], monthly['valence_mean'],
             yerr=monthly['valence_se']*1.96, fmt='none', color='white', capsize=3, lw=1)
ax1.axhline(df['valence'].mean(), color='white', ls='--', lw=1, alpha=0.5, label='media global')
ax1.set_ylabel('valence medio')
ax1.set_title('Valence por mes (↓ = más triste)')
ax1.set_ylim(0, 1)
ax1.grid(axis='y', zorder=0)
ax1.legend(fontsize=8)

legend_patches = [mpatches.Patch(color=c, label=s) for s, c in season_colors.items()]
ax1.legend(handles=legend_patches + [plt.Line2D([0],[0], color='white', ls='--', label='media global')],
           fontsize=8, loc='upper right')

# Panel 2: popularidad por mes
ax2.bar(monthly['month_name'], monthly['popularity_mean'],
        color=bar_colors, alpha=0.9, zorder=2)
ax2.set_ylabel('popularidad media')
ax2.set_title('Popularidad media por mes')
ax2.grid(axis='y', zorder=0)

plt.tight_layout()
plt.savefig('../data/img_estacional.png', dpi=150, bbox_inches='tight')
plt.show()

print('Valence por estación:')
print(df.groupby('season')['valence'].agg(['mean', 'std']).round(3))

## 4. Correlación de Pearson — valence vs popularidad

In [ ]:
clean = df[['valence', 'energy', 'popularity']].dropna()

r_vp, p_vp = stats.pearsonr(clean['valence'], clean['popularity'])
r_ep, p_ep = stats.pearsonr(clean['energy'],  clean['popularity'])

print(f'Pearson valence  vs popularidad: r = {r_vp:+.4f}  (p = {p_vp:.2e})')
print(f'Pearson energy   vs popularidad: r = {r_ep:+.4f}  (p = {p_ep:.2e})')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Correlación de Pearson — audio features vs popularidad', fontsize=13)

for ax, (x_col, color, r, p) in zip(axes, [
    ('valence', PINK,  r_vp, p_vp),
    ('energy',  BLUE,  r_ep, p_ep),
]):
    sample = clean.sample(min(2000, len(clean)), random_state=SEED)
    ax.scatter(sample[x_col], sample['popularity'],
               color=color, alpha=0.25, s=8, edgecolors='none')

    # Línea de regresión
    m, b = np.polyfit(clean[x_col], clean['popularity'], 1)
    x_line = np.linspace(0, 1, 100)
    ax.plot(x_line, m*x_line + b, color='white', lw=1.5)

    sig = '*** (significativo)' if p < 0.001 else ('ns' if p > 0.05 else '* significativo')
    ax.set_title(f'{x_col} vs popularidad\nr = {r:+.3f}  {sig}')
    ax.set_xlabel(x_col)
    ax.set_ylabel('popularidad')
    ax.grid()

plt.tight_layout()
plt.savefig('../data/img_pearson.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Test Mann-Whitney — ¿Es el invierno significativamente más triste?

In [ ]:
winter_val = df[df['season'] == 'Invierno']['valence'].dropna()
summer_val = df[df['season'] == 'Verano']['valence'].dropna()

stat, p_mw = stats.mannwhitneyu(winter_val, summer_val, alternative='less')

print(f'Mann-Whitney U (invierno < verano en valence):')
print(f'  Invierno: media={winter_val.mean():.3f}, n={len(winter_val):,}')
print(f'  Verano:   media={summer_val.mean():.3f}, n={len(summer_val):,}')
print(f'  U={stat:.0f}, p={p_mw:.4f}')
print(f'  Conclusión: {"diferencia significativa" if p_mw < 0.05 else "no hay diferencia significativa"} (α=0.05)')

# Efecto tamaño: rank-biserial correlation
n1, n2 = len(winter_val), len(summer_val)
rb = 1 - (2 * stat) / (n1 * n2)
print(f'  Rank-biserial r = {rb:.4f} (efecto {"pequeño" if abs(rb)<0.3 else "medio" if abs(rb)<0.5 else "grande"})')

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.set_title(f'Invierno vs Verano — distribución de valence\nMann-Whitney p={p_mw:.4f}')

ax.hist(winter_val, bins=40, alpha=0.7, color=BLUE,  label=f'Invierno (n={len(winter_val):,})', density=True)
ax.hist(summer_val, bins=40, alpha=0.7, color=PINK,  label=f'Verano   (n={len(summer_val):,})', density=True)
ax.axvline(winter_val.mean(), color=BLUE, lw=2, ls='--')
ax.axvline(summer_val.mean(), color=PINK, lw=2, ls='--')
ax.set_xlabel('valence')
ax.set_ylabel('densidad')
ax.legend()
ax.grid(axis='y')

plt.tight_layout()
plt.savefig('../data/img_mannwhitney.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Clustering K-Means — ¿Qué emociones agrupan las canciones?

In [ ]:
features_cluster = ['valence', 'energy', 'danceability', 'acousticness']
X = df[features_cluster].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow para k óptimo
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

# Modelo final con k=4 (cuatro cuadrantes emocionales)
K_FINAL = 4
km = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10)
X['cluster'] = km.fit_predict(X_scaled)

# Etiquetar clusters por valence y energy
centers = scaler.inverse_transform(km.cluster_centers_)
centers_df = pd.DataFrame(centers, columns=features_cluster)

def label_cluster(row):
    if row['valence'] >= 0.5 and row['energy'] >= 0.5:   return 'Feliz & Energético'
    if row['valence'] >= 0.5 and row['energy'] < 0.5:    return 'Feliz & Suave'
    if row['valence'] < 0.5  and row['energy'] >= 0.5:   return 'Oscuro & Intenso'
    return 'Triste & Acústico'

centers_df['label'] = centers_df.apply(label_cluster, axis=1)
cluster_labels = {i: centers_df.loc[i, 'label'] for i in range(K_FINAL)}
X['cluster_label'] = X['cluster'].map(cluster_labels)

# Plot
cluster_colors = ['Feliz & Energético', 'Feliz & Suave', 'Oscuro & Intenso', 'Triste & Acústico']
palette = {cluster_labels[i]: c for i, c in zip(range(K_FINAL), [PINK, GREEN, PURPLE, BLUE])}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('K-Means (k=4) — cuatro emociones musicales', fontsize=13)

# Scatter valence vs energy
ax = axes[0]
sample = X.sample(min(3000, len(X)), random_state=SEED)
for label, color in palette.items():
    mask = sample['cluster_label'] == label
    ax.scatter(sample.loc[mask, 'valence'], sample.loc[mask, 'energy'],
               color=color, s=8, alpha=0.4, label=label, edgecolors='none')
ax.set_xlabel('valence (tristeza/alegría)')
ax.set_ylabel('energy')
ax.legend(fontsize=8)
ax.grid()
ax.set_title('Clusters en espacio valence × energy')

# Barras popularidad por cluster
ax = axes[1]
pop_by_cluster = X.join(df[['popularity']]).groupby('cluster_label')['popularity'].mean().sort_values(ascending=False)
bars = ax.barh(pop_by_cluster.index, pop_by_cluster.values,
               color=[palette[l] for l in pop_by_cluster.index], alpha=0.9)
ax.set_xlabel('popularidad media')
ax.set_title('¿Qué emoción es más popular?')
ax.grid(axis='x')
for bar, val in zip(bars, pop_by_cluster.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}',
            va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/img_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCentros de los clusters:')
print(centers_df[['label','valence','energy']].to_string(index=False))

## 7. Conclusiones

In [ ]:
print('=== RESUMEN ===')
print()
print(f'1. Estacionalidad del valence:')
s = df.groupby('season')['valence'].mean().sort_values()
for season, v in s.items():
    print(f'   {season:<12} valence medio = {v:.3f}')

print()
print(f'2. Pearson valence vs popularidad: r = {r_vp:+.3f}')
print(f'   → {"correlación débil" if abs(r_vp)<0.3 else "correlación moderada"}, '
      f'{"p < 0.05 (significativa)" if p_vp < 0.05 else "no significativa"}')

print()
print(f'3. Mann-Whitney invierno vs verano: p = {p_mw:.4f}')
if p_mw < 0.05:
    print('   → Diferencia estadísticamente significativa.')
    print('   → Las canciones lanzadas en invierno tienen valence más bajo.')
else:
    print('   → No hay diferencia significativa entre estaciones.')

print()
print(f'4. Clustering: el grupo más popular es "{pop_by_cluster.index[0]}"')
print(f'   con popularidad media = {pop_by_cluster.iloc[0]:.1f}')